<a href="https://colab.research.google.com/github/ddickson28/Captain-FPSO-BN/blob/Change-to-cumulative-damage/Copy_of_CumulativeDamage17_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mbnpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.3/105.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0

In [ ]:
#Import modules
import numpy as np #importing a module
from mbnpy import variable, cpm, inference #importing relevant code classes from MBNpy
import itertools
from scipy.stats import truncnorm
from math import erf, sqrt

In [ ]:
"Building the BN and relationship"
from mbnpy import variable, cpm, inference #importing relevant code classes from MBNpy

#1 Define variables
# Wx- WeatherDamage Lx - LoadingDamage, Tx - TotalDamage, Px - PreviousRepair, TPrev - TotalDamagePrevious,
# Cx -CumulativeDamage , CLx - CrackLocationInterest,

varis = {}
causes = ['Wx','Lx','Tx']
interventions = ['Px']
effects = ['Cx']
observations = ['Clx']

n_components = 2

for i in range(n_components):
                varis[f'Wx{i+1}'] = variable.Variable(f'Wx{i+1}', ['0','0.2','0.4','0.6','0.8','1.0'] )

for i in range(n_components):
                varis[f'Lx{i+1}'] = variable.Variable(f'Lx{i+1}', ['0','0.2','0.4','0.6','0.8','1.0'] )

for i in range(-1, n_components):
                varis[f'Tx{i+1}'] = variable.Variable(f'Tx{i+1}', ['0','0.2','0.4','0.6','0.8','1.0'] )

for i in range(n_components):
                varis[f'Px{i+1}'] = variable.Variable(f'Px{i+1}', ['False','True'] )

#Does this need to be defined, it will just be the Tx from the previous step
#for i in range(n_components):
#                varis[f'TPrevx{i+1}'] = variable.Variable(f'TPrevx{i+1}', ['0','0.2','0.4','0.6','0.8','1.0'])

for i in range(n_components):
                varis[f'Cx{i+1}'] = variable.Variable(f'Cx{i+1}', ['0','0.2','0.4','0.6','0.8','1.0'])

for i in range(n_components):
                varis[f'CLx{i+1}'] = variable.Variable(f'CLx{i+1}', ['False','True'])


#2 Define the cpm of the variables from 1 in order specified above.

cpms = {}

for i in range(n_components):
                cpms[f'Wx{i+1}'] = cpm.Cpm(
                         variables = [varis[f'Wx{i+1}']], # Use varis dictionary
                            no_child = 1,
                            C=np.array([[0],[1],[2],[3],[4],[5]], dtype=int),
                            p=np.array([0.017,0.435,0.518,0.03,0.0,0.0])
                )

for i in range(n_components):
                cpms[f'Lx{i+1}'] = cpm.Cpm(
                         variables = [varis[f'Lx{i+1}']],
                            no_child = 1,
                            C=np.array([[0],[1],[2],[3],[4],[5]], dtype=int),
                            p=np.array([0.122,0.677,0.198,0.002,0.0,0])
                )

for i in range(n_components):
                cpms[f'Tx{i+1}'] = cpm.Cpm(
                         variables = [varis[f'Tx{i+1}'], varis[f'Wx{i+1}'], varis[f'Lx{i+1}']], # Use varis dictionary for Wx
                            no_child = 1,
                            C=np.array([
                                                [0,0,0],
                                                [1,1,0],
                                                [2,2,0],
                                                [3,3,0],
                                                [4,4,0],
                                                [5,5,0],
                                                [1,0,1],
                                                [2,1,1],
                                                [3,2,1],
                                                [4,3,1],
                                                [5,4,1],
                                                [5,5,1],
                                                [0,0,2],
                                                [3,1,2],
                                                [4,2,2],
                                                [5,3,2],
                                                [5,4,2],
                                                [5,5,2],
                                                [3,0,3],
                                                [4,1,3],
                                                [5,2,3],
                                                [5,3,3],
                                                [5,4,3],
                                                [5,5,3],
                                                [4,0,4],
                                                [5,1,4],
                                                [5,2,4],
                                                [5,3,4],
                                                [5,4,4],
                                                [5,5,4],
                                                [5,0,5],
                                                [2,1,5],
                                                [5,2,5],
                                                [5,3,5],
                                                [5,4,5],
                                                [5,5,5],

                             ], dtype=int),
                             p=np.array([1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,])

)

for i in range(n_components):
                cpms[f'Px{i+1}'] = cpm.Cpm(
                                            variables = [varis[f'Px{i+1}']],
                                            no_child=1,
                                            C=np.array([[0],[1]], dtype=int),
                                            p=np.array([1.0, 0.0])

)

#for i in range(n_components):
#                            cpms[f'TPrevx{i+1}'] = cpm.Cpm(
#                                     variables = [varis[f'TPrevx{i+1}']],
#                                     no_child=1,
#                                     C=np.array([[0],[1],[2],[3],[4],[5]], dtype=int),
#                                     p=np.array([0.0, 0.0, 0.0, 0.0, 0.0, 1.0])

#)

for i in range (n_components):
                            cpms[f'Cx{i+1}'] = cpm.Cpm(
                                    variables = [varis[f'Cx{i+1}'], varis[f'Px{i+1}'], varis[f'Tx{i}'], varis[f'Tx{i+1}']], #Just capture Tx previous step here
                                    no_child=1,
                                    C=np.array([[0,0,0,0],
                                                [0,1,0,0],
                                                [1,0,1,0],
                                                [1,1,1,0],
                                                [2,0,2,0],
                                                [0,1,2,0],
                                                [3,0,3,0],
                                                [0,1,3,0],
                                                [4,0,4,0],
                                                [0,1,4,0],
                                                [5,0,5,0],
                                                [0,1,5,0],
                                                [1,0,0,1],
                                                [1,1,0,1],
                                                [2,0,1,1],
                                                [1,1,1,1],
                                                [3,0,2,1],
                                                [1,1,2,1],
                                                [4,0,3,1],
                                                [1,1,3,1],
                                                [5,0,4,1],
                                                [1,1,4,1],
                                                [5,0,5,1],
                                                [1,1,5,1],
                                                [2,0,0,2],
                                                [2,1,0,2],
                                                [3,0,1,2],
                                                [2,1,1,2],
                                                [4,0,2,2],
                                                [2,1,2,2],
                                                [5,0,3,2],
                                                [2,1,3,2],
                                                [5,0,4,2],
                                                [2,1,4,2],
                                                [5,0,5,2],
                                                [2,1,5,2],
                                                [3,0,0,3],
                                                [3,1,0,3],
                                                [4,0,1,3],
                                                [3,1,1,3],
                                                [5,0,2,3],
                                                [3,1,2,3],
                                                [5,0,3,3],
                                                [3,1,3,3],
                                                [5,0,4,3],
                                                [3,1,4,3],
                                                [5,0,5,3],
                                                [3,1,5,3],
                                                [4,0,0,4],
                                                [4,1,0,4],
                                                [5,0,1,4],
                                                [4,1,1,4],
                                                [5,0,2,4],
                                                [4,1,2,4],
                                                [5,0,3,4],
                                                [4,1,3,4],
                                                [5,0,4,4],
                                                [4,1,4,4],
                                                [5,0,5,4],
                                                [4,1,5,4],
                                                [5,0,0,5],
                                                [5,1,0,5],
                                                [5,0,1,5],
                                                [5,1,1,5],
                                                [5,0,2,5],
                                                [5,1,2,5],
                                                [5,0,3,5],
                                                [5,1,3,5],
                                                [5,0,4,5],
                                                [5,1,4,5],
                                                [5,0,5,5],
                                                [5,1,5,5],
                        ], dtype=int),
                            p=np.array([1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])
)

for i in range (n_components):
                            cpms[f'CLx{i+1}'] = cpm.Cpm(
                                     variables = [varis[f'CLx{i+1}'], varis[f'Cx{i+1}']],
                                     no_child=1,
                                     C=np.array([[0,0],
                                                 [0,1],
                                                 [0,2],
                                                 [0,3],
                                                 [0,4],
                                                 [0,5],
                                                 [1,0],
                                                 [1,1],
                                                 [1,2],
                                                 [1,3],
                                                 [1,4],
                                                 [1,5],
                                 ], dtype=int),
                                 p=np.array([1.0, 1.0, 1.0, 0.99, 0.83, 0.18, 0.0, 0.0, 0.0, 0.01, 0.17, 0.82])
                            )


print(varis)
print(cpms)
#marg_CrackLocationInterest = inference.variable_elim(cpms = [cpms_WeatherDamage, cpms_LoadingDamage, cpms_TotalDamage, cpms_PreviousRepair, cpms_TotalDamage_Prev, cpms_CummulativeDamage, cpms_CrackLocationInterest],
#                                                     var_elim = ['WeatherDamage', 'LoadingDamage', 'TotalDamage', 'PreviousRepair','TotalDamage_Prev', 'CummulativeDamage', ],
#                                                                                                               prod= True )

#print(marg_CrackLocationInterest)



{'Wx1': <Variable representing Wx1['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x799384769430>, 'Wx2': <Variable representing Wx2['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x79937f161820>, 'Lx1': <Variable representing Lx1['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x79937f163c20>, 'Lx2': <Variable representing Lx2['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x79937f0efb30>, 'Tx0': <Variable representing Tx0['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x79937f0efce0>, 'Tx1': <Variable representing Tx1['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x79937f0ef680>, 'Tx2': <Variable representing Tx2['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x79937f0eed50>, 'Px1': <Variable representing Px1['False', 'True'] at 0x79937f0edeb0>, 'Px2': <Variable representing Px2['False', 'True'] at 0x79937f0eef60>, 'Cx1': <Variable representing Cx1['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x79937f0ee660>, 'Cx2': <Variable representing Cx2['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x79937ef96b70>, 'CLx1': <Va

In [ ]:
varis = {} # create an empty dictionary called varis

causes = ['Wx', 'Lx', 'Tx'] # Creates a list of 3 strings

n_components = 6
for i in range(n_components):
        varis[f'Wx{i+1}'] = variable.Variable(f'Wx{i+1}', ['0','0.2','0.4','0.6','0.8','1.0'] ) # creates Wx1 to Wx6

for i in range(n_components):
        varis[f'Lx{i+1}'] = variable.Variable(f'Lx{i+1}', ['0','0.2','0.4','0.6','0.8','1.0'] ) # creates Lx1 to Lx6

for i in range(n_components):
        varis[f'Tx{i+1}'] = variable.Variable(f'Tx{i+1}', ['0','0.2','0.4','0.6','0.8','1.0'] ) # creates Tx1 to Tx6

varis_elim_order = [varis[f"{c}{i}"] for c in causes for i in range(n_components) ] # Create a varis_elim_order list...

cpms = {}
cpms[f'Wx{i+1}'] = cpm.Cpm(
             [WeatherDamage], no_child=1,
               C=np.array([[0],[1],[2],[3],[4],[5]], dtype=int),
               p=np.array([0.017,0.435,0.518,0.03,0.0,0.0])
)